# 03 · Train the GPTs and run the experiments (GPU)

**Runtime → Change runtime type → T4 GPU.** Needs notebooks 01 (corpus) and 02 (tokenizers)
done first: it reads `splits/` and `tokenizers/` from your Drive.

| Run | What | Rough time on a free T4 |
|---|---|---|
| `hinglish16k-5m` | smallest model | ~10 min |
| `hinglish16k-15m` | medium | ~45 min |
| `hinglish16k-30m` | largest | ~3 h |
| `english16k-15m` | same 15M model, English-only tokenizer (the BPB baseline) | ~45 min |

**If Colab disconnects:** reconnect, run the setup cells (1-4) again, then re-run the training
cell that was interrupted. It continues from the last checkpoint on Drive (saved every 250
steps, about every 2-5 minutes). Finished runs are skipped instantly.

## 1. Setup

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    KEEP = "/content/drive/MyDrive/hinglish-nanolm/data"   # corpus + tokenizers (from 01, 02)
    RUNS = "/content/drive/MyDrive/hinglish-nanolm/runs"   # checkpoints + logs survive here
    DATA = "/content/data"                                 # fast local copy for training
else:
    KEEP = DATA = os.path.abspath("../data")
    RUNS = os.path.abspath("../runs")
os.makedirs(RUNS, exist_ok=True)
os.makedirs(DATA, exist_ok=True)
if KEEP != DATA and not os.path.exists(f"{DATA}/splits"):
    os.symlink(f"{KEEP}/splits", f"{DATA}/splits")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = "https://github.com/naqi7272/Hinglish-nanoLM.git"
BRANCH = "claude/hinglish-nanolm-setup-finaye"
if IN_COLAB:
    %cd /content
    !git clone -q -b {BRANCH} {REPO} code || (cd code && git pull -q)
    %cd /content/code
    !pip install -q -e ".[wandb]"

Optional: Weights & Biases dashboards. Add your key (from https://wandb.ai/authorize) in
Colab's 🔑 Secrets as `WANDB_API_KEY`. Without it, everything is still logged to CSV files
and all plots work.

In [ ]:
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("W&B on")
except Exception:
    print("W&B off (CSV logging only)")

## 2. Tokenize the corpus into token files (~10-20 min once; then reused from Drive)

In [ ]:
import shutil
os.makedirs(f"{DATA}/tokenizers", exist_ok=True)
for name in ("hinglish16k", "english16k"):
    shutil.copy(f"{KEEP}/tokenizers/{name}.json", f"{DATA}/tokenizers/{name}.json")
    if os.path.exists(f"{KEEP}/bins/{name}/meta.json"):          # packed in an earlier session
        shutil.copytree(f"{KEEP}/bins/{name}", f"{DATA}/bins/{name}", dirs_exist_ok=True)
    !python scripts/pack_data.py --tokenizer {name} --data-dir {DATA}
    shutil.copytree(f"{DATA}/bins/{name}", f"{KEEP}/bins/{name}", dirs_exist_ok=True)

## 3. Smoke test (1 min)
A tiny model for 40 steps on the real data: checks the whole pipeline on the GPU before
the long runs. The loss should go down and you should see `eval` lines.

In [ ]:
!python scripts/train.py --config configs/train/smoke_cpu.yaml --data-dir {DATA} --runs-dir /content/smoke --device cuda

## 4. Training runs (re-run a cell after a disconnect to resume it)

In [ ]:
!python scripts/train.py --config configs/train/hinglish16k-5m.yaml --data-dir {DATA} --runs-dir {RUNS}

In [ ]:
!python scripts/train.py --config configs/train/hinglish16k-15m.yaml --data-dir {DATA} --runs-dir {RUNS}

In [ ]:
# Tokenizer baseline (cheap version): same 5M model as hinglish16k-5m, English-only tokenizer.
# The 15M version (configs/train/english16k-15m.yaml) is optional if you have GPU time.
!python scripts/train.py --config configs/train/english16k-5m.yaml --data-dir {DATA} --runs-dir {RUNS}


In [ ]:
# OPTIONAL: the 30M run takes many hours on a free T4. Skip it unless you have spare GPU time.
# !python scripts/train.py --config configs/train/hinglish16k-30m.yaml --data-dir {DATA} --runs-dir {RUNS}


## 5. KV-cache speed, samples, plots

In [ ]:
!python scripts/bench_generate.py --out results/kv_cache_speed.json
for run in ("hinglish16k-5m", "hinglish16k-15m", "hinglish16k-30m", "english16k-15m", "english16k-5m"):
    if os.path.exists(f"{RUNS}/{run}/final.pt"):
        !python scripts/sample.py --run {RUNS}/{run} --data-dir {DATA}
!python scripts/make_plots.py --runs-dir {RUNS} --out plots --results results

In [ ]:
from IPython.display import Image, display
for p in ("loss_vs_tokens", "loss_vs_size", "bpb_tokenizers", "kv_cache_speed"):
    if os.path.exists(f"plots/{p}.png"):
        display(Image(f"plots/{p}.png"))

## 6. Save results
Downloads `m5_results.zip` (plots, numbers, samples, training logs; no big checkpoints).
Unzip it inside your `Hinglish-nanoLM` folder and push.

In [ ]:
import glob
os.makedirs("results/runs", exist_ok=True)
for run_dir in glob.glob(f"{RUNS}/*/"):
    name = os.path.basename(run_dir.rstrip("/"))
    os.makedirs(f"results/runs/{name}", exist_ok=True)
    for f in ("config.json", "train.csv", "eval.csv"):
        if os.path.exists(f"{run_dir}/{f}"):
            shutil.copy(f"{run_dir}/{f}", f"results/runs/{name}/{f}")
shutil.copytree("results", f"{KEEP}/../m5/results", dirs_exist_ok=True)
shutil.copytree("plots", f"{KEEP}/../m5/plots", dirs_exist_ok=True)
!zip -qr m5_results.zip results plots
if IN_COLAB:
    from google.colab import files
    files.download("m5_results.zip")